In [5]:
import time
import re
import traceback
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

In [6]:
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--disable-notifications")
options.add_argument("--no-sandbox")

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

In [7]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.replace("______________________________________________________________________", "")
    text = text.replace("|", " ")
    text = text.replace("●", " ")
    return text.strip()

In [8]:
input_excel = r"D:/BaiDoAnChuyenNganh3/Automated-Resume-Ranking-System-main/csvfiles/crawlcv/re_naukri_links.xlsx"
df_links = pd.read_excel(input_excel)

In [9]:
results = []

In [10]:
for idx, row in df_links.iterrows():
    link = str(row["Links"]).strip()
    category = str(row["Category"]).strip()
    print(f"\n👉 Đang truy cập link {idx+1}/{len(df_links)}: {link}")

    try:
        driver.get(link)
        time.sleep(3)  

        soup = BeautifulSoup(driver.page_source, "html.parser")

        resume_divs = soup.find_all("div", class_="resume-div")
        for div_index, div in enumerate(resume_divs, start=1):
            html_content = str(div)
            text_content = clean_text(div.get_text(separator=" ", strip=True))
            results.append({
                "Source_Link": link,
                "Category": category,
                "Resume_Index": f"div_{div_index}",
                "Resume_HTML": html_content,
                "Resume_Text": text_content
            })
            print(f"   -> Lấy được resume-div #{div_index} ({len(text_content)} ký tự)")

        tables = soup.find_all("table", attrs={
            "border": "1",
            "style": lambda v: v and "border-collapse: collapse" in v
        })

        for table_index, table in enumerate(tables, start=1):
            tbody = table.find("tbody")
            if tbody:
                html_content = str(tbody)
                text_content = clean_text(tbody.get_text(separator=" ", strip=True))
                results.append({
                    "Source_Link": link,
                    "Category": category,
                    "Resume_Index": f"table_{table_index}",
                    "Resume_HTML": html_content,
                    "Resume_Text": text_content
                })
                print(f"   -> Lấy được table #{table_index} ({len(text_content)} ký tự)")

    except Exception as e:
        print(f"⚠️ Lỗi khi xử lý link: {link}")
        traceback.print_exc()
        results.append({
            "Source_Link": link,
            "Category": category,
            "Resume_Index": "ERROR",
            "Resume_HTML": "",
            "Resume_Text": "",
            "Error": str(e)
        })


👉 Đang truy cập link 1/16: https://www.naukri.com/career-advice/graphic-designer-resume-sample-ffid
   -> Lấy được resume-div #1 (4552 ký tự)
   -> Lấy được table #1 (975 ký tự)
   -> Lấy được table #2 (1237 ký tự)
   -> Lấy được table #3 (1702 ký tự)
   -> Lấy được table #4 (210 ký tự)
   -> Lấy được table #5 (208 ký tự)
   -> Lấy được table #6 (303 ký tự)
   -> Lấy được table #7 (80 ký tự)

👉 Đang truy cập link 2/16: https://www.naukri.com/career-advice/network-engineer-resume-sample-ffid
   -> Lấy được resume-div #1 (3303 ký tự)
   -> Lấy được table #1 (1530 ký tự)
   -> Lấy được table #2 (1593 ký tự)
   -> Lấy được table #3 (1940 ký tự)
   -> Lấy được table #4 (293 ký tự)
   -> Lấy được table #5 (80 ký tự)

👉 Đang truy cập link 3/16: https://www.naukri.com/career-advice/web-designer-resume-sample-ffid
   -> Lấy được resume-div #1 (3170 ký tự)
   -> Lấy được table #1 (126 ký tự)
   -> Lấy được table #2 (154 ký tự)
   -> Lấy được table #3 (80 ký tự)

👉 Đang truy cập link 4/16: https

In [11]:
driver.quit()

In [13]:
output_excel = r"D:\BaiDoAnChuyenNganh3\Automated-Resume-Ranking-System-main\csvfiles\crawlcv\final_data_naukri_resume.xlsx"

if results:
    df_out = pd.DataFrame(results)
    df_out.to_excel(output_excel, index=False)
    print(f"\n✅ Đã lưu {len(results)} dòng dữ liệu vào file:\n{output_excel}")
else:
    print("\n⚠️ Không có dữ liệu nào được crawl!")


✅ Đã lưu 99 dòng dữ liệu vào file:
D:\BaiDoAnChuyenNganh3\Automated-Resume-Ranking-System-main\csvfiles\crawlcv\final_data_naukri_resume.xlsx
